# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# Clone the FlyRank internship starter repository
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [11]:
# Check the FlyRank raw data folder
import os

data_path = "flyrank-ml-internship-starter/data/raw"

print("Files in data/raw:")
print(os.listdir(data_path))

Files in data/raw:
['content_refresh_anonymized.csv']


In [12]:
# Load the FlyRank content refresh dataset
import pandas as pd

file_path = "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

# Check dataset size and available columns
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Build the feature vector

### Feature Vector

I will use observed page-level search performance signals to build the feature vector. The initial features are based on impressions, clicks, CTR, average search position, search volume, and page age. Missing numeric values will be handled explicitly, and categorical fields will be encoded where needed. Features are included only when they are available before the prediction moment.


In [13]:
# Build the feature vector for content opportunity analysis

# Numeric features available before the prediction moment
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

# Categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

# Keep only the selected features
X = df[numeric_features + categorical_features].copy()

# Fill missing numeric values with the median
for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

# Fill missing categorical values with an explicit category
for col in categorical_features:
    X[col] = X[col].fillna("missing").astype(str)

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=categorical_features,
    dummy_na=False
)

print("Original rows:", len(df))
print("Feature vector shape:", X.shape)
print("Missing values remaining:", X.isna().sum().sum())
print("\nFeature vector preview:")
display(X.head())

Original rows: 30000
Feature vector shape: (30000, 70)
Missing values remaining: 0

Feature vector preview:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,engaged_sessions_90d,...,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,trend_direction_down,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,1,...,False,False,False,True,False,True,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,0,...,False,False,True,False,False,True,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,0,...,False,False,True,False,False,True,False,False,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,1,...,False,True,False,False,False,False,False,False,True,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,0,...,False,False,True,False,False,True,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)
# Build the feature vector for content opportunity analysis

# Numeric features available before the prediction moment
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

# Categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

# Keep only the selected features
X = df[numeric_features + categorical_features].copy()

# Fill missing numeric values with the median
for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

# Fill missing categorical values with an explicit category
for col in categorical_features:
    X[col] = X[col].fillna("missing").astype(str)

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=categorical_features,
    dummy_na=False
)

print("Original rows:", len(df))
print("Feature vector shape:", X.shape)
print("Missing values remaining:", X.isna().sum().sum())
print("\nFeature vector preview:")
display(X.head())

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check that the selected feature vector has no missing values
print("Feature vector shape:", X.shape)
print("Total missing values:", X.isna().sum().sum())

# Show the data types after categorical encoding
print("\nFeature vector data types:")
print(X.dtypes.value_counts())


Feature vector shape: (30000, 70)
Total missing values: 0

Feature vector data types:
bool       44
int64      15
float64    11
Name: count, dtype: int64


## 3.Leakage Hunt

I checked the selected features for possible leakage.

I excluded identifiers such as `content_id` and `client_id` because they identify records or clients rather than describe the page opportunity.

I also excluded model/provider fields such as `provider_used` and `model_used` because they describe how the data or content was produced, not the underlying page opportunity.

I treated historical performance features as valid only when their observation window ends before the prediction moment. I did not use future performance or post-outcome information.

Tier features were checked because some are derived from other variables. They are retained as representations of already-observed signals, but they must use the same historical cutoff as their source variables.

No feature should be created using the target or any information that becomes available after the prediction moment.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Leakage check: identify columns that should not enter the feature vector

excluded_columns = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

# Check that excluded columns are not present in the feature vector
present_excluded = [col for col in excluded_columns if col in X.columns]

print("Excluded columns found in feature vector:", present_excluded)

# Check for suspicious outcome/future-related column names
suspicious_keywords = [
    "label",
    "target",
    "outcome",
    "future",
    "next",
    "post"
]

suspicious_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("\nPotentially suspicious columns in raw dataset:")
print(suspicious_columns)

# Final basic check
print("\nRows in raw dataset:", len(df))
print("Rows in feature vector:", len(X))


Excluded columns found in feature vector: []

Potentially suspicious columns in raw dataset:
[]

Rows in raw dataset: 30000
Rows in feature vector: 30000


### Excluded Fields

| Field           | Reason for exclusion                                                                                            |
| --------------- | --------------------------------------------------------------------------------------------------------------- |
| `content_id`    | Identifier only; it does not describe content opportunity and may cause the model to memorize individual pages. |
| `client_id`     | Identifier only; it can create client-specific patterns instead of learning generalizable page signals.         |
| `provider_used` | Describes the data/content provider rather than the page's search opportunity.                                  |
| `model_used`    | Describes the model used to produce content rather than the page's observed opportunity.                        |

I also exclude any target, label, future-window, or post-outcome field if introduced later. These fields could reveal information that would not be available at the prediction moment and could create feature leakage.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify that excluded fields are not part of the feature vector

excluded_columns = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

# The feature vector uses only the selected feature columns
print("Excluded fields:")
for col in excluded_columns:
    print(f"- {col}")

# Verify none of the excluded fields were included
leakage_check = [
    col for col in excluded_columns
    if col in X.columns
]

print("\nExcluded fields present in feature vector:", leakage_check)

if len(leakage_check) == 0:
    print("Check passed: excluded fields are not in the feature vector.")
else:
    print("Warning: an excluded field is present in the feature vector.")


Excluded fields:
- content_id
- client_id
- provider_used
- model_used

Excluded fields present in feature vector: []
Check passed: excluded fields are not in the feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.